# SBI tuning for Local Group catalogue inference

This notebook is a config-driven tuning experiment for the wrapped SBI workflow.

Selection and filter cuts, the active observed features, and the single active target are loaded from `config/config.json` via `config.loader.load_config()`.

Change the active feature flags or switch the single active target in that file, rerun from the top, and the notebook will tune NPE for the new setup.


In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = next((path for path in [cwd, *cwd.parents] if (path / "analogues").is_dir()), None)
if repo_root is None:
    raise FileNotFoundError("Could not find the repo root containing the `analogues` package.")

candidate_paths = [
    repo_root,
    repo_root / "scripts",
    repo_root / "illustris_python",
    Path("/mnt/data"),
]
for pth in candidate_paths:
    if pth.exists() and str(pth) not in sys.path:
        sys.path.insert(0, str(pth))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
import illustris_python as il

from analogues import AnalogueSample, FilterConfig, SelectionConfig
from config.loader import load_config
from lg_sbi import infer_with_npe, plot_expected_coverage, plot_sbc_rank_hist, prepare_sbi_problem
from lg_sbi_practical_guide import plot_1d_posterior_comparison
from lg_sbi_tuning import run_sbi_optuna_study

print("Repo root:", repo_root)
print("Imports loaded successfully.")


## 1) Load the simulation data and rebuild the Local Group pair catalogue

This setup uses the same analogue-generation workflow as `generate_analogues.ipynb`, together with the shared config loader. The selection and filter settings come directly from `config/config.json`.


In [ ]:
def resolve_base_path(repo_root: Path) -> str:
    raw_candidates = [
        os.environ.get("TNG_BASE_PATH"),
        os.environ.get("ILLUSTRIS_BASE_PATH"),
        str(repo_root / "tng300" / "outputs"),
    ]

    for raw in raw_candidates:
        if not raw:
            continue
        base = Path(raw).expanduser()
        for candidate in (base, base / "tng300" / "outputs"):
            if (candidate / "groups_099").exists():
                return str(candidate.resolve())

    raise FileNotFoundError(
        "Could not find a valid TNG outputs directory. Set TNG_BASE_PATH or ILLUSTRIS_BASE_PATH "
        "to the outputs directory, or to a root that contains tng300/outputs."
    )


config_loader = load_config()
analysis_config = config_loader.get_config()

BASE_PATH = resolve_base_path(repo_root)
SNAP = 99

selection_config = SelectionConfig(**analysis_config.selection)
filter_config = FilterConfig(**analysis_config.filter)

print("Using BASE_PATH:", BASE_PATH)
print("SNAP =", SNAP)
print("Selection config:")
display(pd.DataFrame([analysis_config.selection]))
print("Filter config:")
display(pd.DataFrame([analysis_config.filter]))

analogue_sample = AnalogueSample(
    base_path=BASE_PATH,
    selection_config=selection_config,
    filter_config=filter_config,
    verbose=True,
    snap=SNAP,
)

sub = analogue_sample._sub
sample = analogue_sample._sample
pair_set = analogue_sample._pair_set
pipeline = analogue_sample._pipeline
halos = il.groupcat.loadHalos(BASE_PATH, SNAP, fields=["Group_M_Crit200"])
h = analogue_sample._selection_config.hubble_param

pairs = analogue_sample.pairs
pair_i = pairs.i
pair_j = pairs.j

mstar_i = sample.m_stellar[pair_i].astype(np.float64, copy=False)
mstar_j = sample.m_stellar[pair_j].astype(np.float64, copy=False)
mdm_i = sample.m_dark_matter[pair_i].astype(np.float64, copy=False)
mdm_j = sample.m_dark_matter[pair_j].astype(np.float64, copy=False)

catalog = {
    "r_kpc": pairs.separation.astype(np.float64, copy=False),
    "v_r": pairs.vr.astype(np.float64, copy=False),
    "v_t": pairs.vt.astype(np.float64, copy=False),
    "same_host": pairs.have_same_host.astype(np.float64, copy=False),
    "mstar_i": mstar_i,
    "mstar_j": mstar_j,
    "mstar_big": np.maximum(mstar_i, mstar_j),
    "mstar_small": np.minimum(mstar_i, mstar_j),
    "mdm_i": mdm_i,
    "mdm_j": mdm_j,
    "mdm_sum": mdm_i + mdm_j,
}

catalog["mdm_big"] = np.maximum(catalog["mdm_i"], catalog["mdm_j"])
catalog["mdm_small"] = np.minimum(catalog["mdm_i"], catalog["mdm_j"])
catalog["dm_mass_ratio"] = catalog["mdm_big"] / np.clip(catalog["mdm_small"], 1e-30, None)
catalog["mstar_sum"] = catalog["mstar_i"] + catalog["mstar_j"]

selected_group_numbers = np.asarray(sample.grnr, dtype=int)
group_i = selected_group_numbers[pair_i]
group_j = selected_group_numbers[pair_j]

if isinstance(halos, dict):
    group_m200c_native = np.asarray(halos["Group_M_Crit200"], dtype=float)
else:
    group_m200c_native = np.asarray(halos, dtype=float)
group_m200c_msun = group_m200c_native * 1.0e10 / h

m200c_i = group_m200c_msun[group_i]
m200c_j = group_m200c_msun[group_j]

catalog["m200c_i"] = m200c_i
catalog["m200c_j"] = m200c_j
catalog["m200c_big"] = np.maximum(m200c_i, m200c_j)
catalog["m200c_small"] = np.minimum(m200c_i, m200c_j)
catalog["m200c_sum"] = m200c_i + m200c_j
catalog["m200c_ratio"] = catalog["m200c_big"] / np.clip(catalog["m200c_small"], 1e-30, None)

display(pd.DataFrame(pipeline.get_cutflow()))
print("Loaded subhalos:", sub["count"])
print("Selected objects:", sample.keep_idx.size)
print("Pairs before filtering:", pair_set.i.size)
print("Pairs after filtering:", pairs.i.size)


## 2) Load the active features and target from the shared config

The active feature set and the single active target come from `config/config.json`. The notebook-local values below only control the observation override and the tuning budget.


In [ ]:
FEATURE_INFO = analysis_config.features
ACTIVE_FEATURES = list(FEATURE_INFO.keys())
if not ACTIVE_FEATURES:
    raise ValueError("config/config.json must have at least one active feature.")

TARGET_KEY = analysis_config.target_name
TARGET = analysis_config.target["column"]
TARGET_LOG10 = bool(analysis_config.target.get("log10", False))
TARGET_LABEL = analysis_config.target.get("label", TARGET_KEY)
LOG10_FEATURES = {
    name for name, meta in FEATURE_INFO.items()
    if bool(meta.get("log10", False))
}

X_OBS_OVERRIDE = {}
TUNING_TRIALS = 25
POSTERIOR_SAMPLES = 20_000

x_obs = np.array(
    [float(X_OBS_OVERRIDE.get(name, FEATURE_INFO[name]["obs"])) for name in ACTIVE_FEATURES],
    dtype=float,
)
sigma = np.array([float(FEATURE_INFO[name]["sigma"]) for name in ACTIVE_FEATURES], dtype=float)

problem = prepare_sbi_problem(
    catalog=catalog,
    features=ACTIVE_FEATURES,
    target=TARGET,
    log10_features=LOG10_FEATURES,
    target_log10=TARGET_LOG10,
)

X = problem["X"]
theta = problem["theta"]
feature_names = problem["feature_names"]
target_name = TARGET_LABEL

display(pd.DataFrame([
    {
        "feature": name,
        "x_obs": FEATURE_INFO[name]["obs"],
        "sigma": FEATURE_INFO[name]["sigma"],
        "label": FEATURE_INFO[name].get("label", name),
        "description": FEATURE_INFO[name].get("description", ""),
    }
    for name in feature_names
]))
display(config_loader.show_active_target())
print("Active target key:", TARGET_KEY)
print("Target column:", TARGET)
print("Target label:", TARGET_LABEL)
print("Target log10 =", TARGET_LOG10)
print("Training rows:", X.shape[0])
print("Dropped rows:", problem["dropped_rows"])
print("x_obs =", x_obs)
print("sigma =", sigma)


## 3) Run the Optuna-based SBI tuning study


In [ ]:
tuning_result = run_sbi_optuna_study(
    X=X,
    theta=theta,
    x_obs=x_obs,
    feature_names=feature_names,
    target_name=target_name,
    n_trials=TUNING_TRIALS,
)
best_fit = tuning_result["best_fit"]
trials_df = tuning_result["trials_df"].sort_values("value", ascending=False, na_position="last")

display(pd.DataFrame([{
    "best_value": tuning_result["best_value"],
    **tuning_result["best_params"],
}]))
display(trials_df)


In [ ]:
tuning_result["best_params"]

## 4) Draw samples from the best-tuned posterior


In [ ]:
sbi_result = infer_with_npe(
    best_fit,
    x_obs=x_obs,
    num_samples=POSTERIOR_SAMPLES,
    random_state=0,
)

display(pd.DataFrame([{"method": "Best tuned SBI", **sbi_result["summary"]}]))
fig, ax = plot_1d_posterior_comparison(
    theta,
    sbi_result["samples"],
    target_name=target_name,
    title=f"Best-tuned NPE posterior for {target_name}",
)
plt.show()


## 5) Tuned SBI diagnostic summary


In [ ]:
diag = best_fit["diagnostics"]
display(pd.DataFrame([{
    "train_size": diag["train_size"],
    "calibration_size": diag["calibration_size"],
    "mean_calibration_log_prob": diag["mean_calibration_log_prob"],
}]))

support = diag.get("x_obs_support_check", {})
if support:
    display(pd.DataFrame({
        "feature": feature_names,
        "x_obs": x_obs,
        "train_min": support["feature_min"],
        "train_max": support["feature_max"],
        "outside_range": support["outside_range"],
    }))
    print("Nearest standardized distance:", support["nearest_distance"])

if diag["sbc_ranks"].size > 0:
    plot_sbc_rank_hist(
        diag["sbc_ranks"],
        num_posterior_samples=diag["sbc_summary"]["num_posterior_samples"],
        parameter_label=target_name,
        title="Best-tuned SBI SBC ranks",
    )
    plt.show()

coverage_result = diag.get("expected_coverage_1d", {})
if len(np.asarray(coverage_result.get("levels", []))) > 0:
    plot_expected_coverage(coverage_result, title="Best-tuned SBI expected coverage")
    plt.show()
